# Передать архивы из Google Drive без скачивания на телефон
Открой этот блокнот в Google Colab и запусти единственную ячейку ниже. Разреши доступ к своему Google Drive.

Блокнот скачает два заданных архива во временную среду Colab, проверит их SHA-256 и создаст в твоём Drive новую **закрытую** папку с частями по 90 МиБ и манифестом. Исходники не изменяются. Потребуется примерно 1,7 ГБ дополнительного места на Drive. Дождись надписи ГОТОВО и передай агенту ссылку на папку. Не запускай несколько копий одновременно. При повторном запуске создаётся новая папка; неполные папки можно удалить вручную.

Никакие пароли, ключи или токены копировать в чат не нужно. Этот блокнот не меняет сетевые настройки Arena и не публикует файлы. Он использует обычный доступ Colab к Drive и размер частей в пределах лимита инструмента.

Время выполнения зависит от скорости Google Drive. Если выполнение прервано, папка может быть неполной: готовность подтверждает только сообщение ГОТОВО после загрузки всех частей и манифеста.

In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, tempfile

SOURCES = [
    dict(id='1b-kDHxNE0aeqpMtLH_iRzdBd2syb0zmp', name='assets.zip', size=1264737618, sha256='cabff06b986e57683a45607690cfa18ce4803a455396e6e923a2c952ed0fff13'),
    dict(id='17K21PujL-qA8XYgl5F6MiofwPRgE2nAO', name='blender-5.3.0-linux-x64.tar.xz', size=406944100, sha256='dacb8264197a3f7fda6001285a5a506fc97ee943881261c2c3bd63c882f22e05'),
]
PART_SIZE = 90 * 1024 * 1024
credentials, _ = google.auth.default()
drive = build('drive', 'v3', credentials=credentials, cache_discovery=False)

def upload(path, folder, mime='application/octet-stream'):
    request = drive.files().create(
        body={'name': path.name, 'parents': [folder]},
        media_body=MediaFileUpload(str(path), mimetype=mime, chunksize=8*1024*1024, resumable=True),
        fields='id,name,size')
    result = None
    while result is None:
        _, result = request.next_chunk(num_retries=5)
    if int(result['size']) != path.stat().st_size:
        raise RuntimeError('Размер загруженной части не совпал: ' + path.name)
    return result['id']

with tempfile.TemporaryDirectory(prefix='asset-transfer-') as tmp:
    root = Path(tmp)
    # Сначала скачиваем и проверяем ОБА оригинала, только потом создаём части.
    for source in SOURCES:
        metadata = drive.files().get(fileId=source['id'], fields='id,name,size', supportsAllDrives=True).execute()
        if int(metadata['size']) != source['size']:
            raise RuntimeError('Размер оригинала изменился: ' + source['name'])
        path = root / source['name']
        print('Скачивание в Colab:', source['name'], flush=True)
        with path.open('wb') as stream:
            downloader = MediaIoBaseDownload(stream, drive.files().get_media(fileId=source['id'], supportsAllDrives=True), chunksize=16*1024*1024)
            done, last_percent = False, -10
            while not done:
                status, done = downloader.next_chunk(num_retries=5)
                percent = int(status.progress()*100)
                if percent >= last_percent + 10 or done:
                    print(f'  {percent}%', flush=True)
                    last_percent = percent
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(8*1024*1024), b''):
                digest.update(chunk)
        if path.stat().st_size != source['size'] or digest.hexdigest() != source['sha256']:
            raise RuntimeError('Проверка SHA-256 не пройдена: ' + source['name'])
        print('SHA-256 совпал:', source['name'], flush=True)

    folder = drive.files().create(body={
        'name': 'Arena asset parts ' + datetime.now(timezone.utc).strftime('%Y-%m-%d %H-%M-%S UTC'),
        'mimeType': 'application/vnd.google-apps.folder'
    }, fields='id').execute()['id']
    manifest = {'format': 'raw-concatenation-v1', 'part_size': PART_SIZE, 'sources': []}
    for source in SOURCES:
        entry = dict(source, parts=[])
        with (root / source['name']).open('rb') as stream:
            index = 1
            while True:
                data = stream.read(PART_SIZE)
                if not data:
                    break
                part = root / (source['name'] + f'.part{index:03d}')
                part.write_bytes(data)
                print('Загрузка на Drive:', part.name, flush=True)
                part_id = upload(part, folder)
                entry['parts'].append({'name': part.name, 'id': part_id, 'size': len(data), 'sha256': hashlib.sha256(data).hexdigest()})
                part.unlink()
                index += 1
        manifest['sources'].append(entry)
    manifest_path = root / 'transfer-manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    upload(manifest_path, folder, 'application/json')
    print('\nГОТОВО. Все части и манифест загружены. Отправь агенту эту ссылку:', flush=True)
    print('https://drive.google.com/drive/folders/' + folder)
